# # Lynceus Fraud Detection System

# ## Production Model Export & Deployment Verification

This notebook validates all production artifacts generated during training and evaluation.

It simulates the exact inference pipeline that will be used by the FastAPI backend.

### Objectives

- Verify saved artifacts
- Load preprocessing pipeline
- Load trained model
- Load metadata
- Run end-to-end prediction
- Validate deployment readiness

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import json
import joblib

import pandas as pd

import numpy as np

In [2]:
DATA_DIR = Path("/Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/data/synthetic")

ARTIFACT_DIR = Path("/Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/artifacts")

TEST_PATH = DATA_DIR / "transactions_test.csv"

PIPELINE_PATH = ARTIFACT_DIR / "preprocessing_pipeline.joblib"

MODEL_PATH = ARTIFACT_DIR / "best_model.joblib"

METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"

In [3]:
# verifying the files

required_files = [

    PIPELINE_PATH,

    MODEL_PATH,

    METADATA_PATH

]

print("=" * 60)
print("Checking Production Artifacts")
print("=" * 60)

for file in required_files:

    if file.exists():

        print(f"✓ {file.name}")

    else:

        raise FileNotFoundError(file)

Checking Production Artifacts
✓ preprocessing_pipeline.joblib
✓ best_model.joblib
✓ model_metadata.json


In [4]:
# loading everything!

preprocessor = joblib.load(
    PIPELINE_PATH
)

model = joblib.load(
    MODEL_PATH
)

with open(METADATA_PATH) as f:

    metadata = json.load(f)

print(type(preprocessor))
print(type(model))

metadata

<class 'sklearn.compose._column_transformer.ColumnTransformer'>
<class 'lightgbm.sklearn.LGBMClassifier'>


{'model_name': 'LGBMClassifier',
 'model_version': '1.0.0',
 'optimal_threshold': 0.4000000000000001,
 'feature_count': 41,
 'validation_metrics': {'accuracy': 0.9845333333333334,
  'precision': 0.7058823529411765,
  'recall': 0.5341246290801187,
  'f1_score': 0.6081081081081081,
  'roc_auc': 0.8837575592981062,
  'pr_auc': 0.5836839758163083},
 'test_metrics': {'accuracy': 0.9834,
  'precision': 0.7414634146341463,
  'recall': 0.4367816091954023,
  'f1_score': 0.5497287522603979,
  'roc_auc': 0.8663020387158318,
  'pr_auc': 0.5640666643027727}}

In [5]:
# load raw test data

test_df = pd.read_csv(

    TEST_PATH,

    parse_dates=["timestamp"]

)

print(test_df.shape)

display(test_df.head())

(15000, 23)


,timestamp,amount,currency,payment_method,hour,day_of_week,is_weekend,sender_account_age_days,receiver_account_age_days,sender_txn_count_24h,...,is_new_receiver,origin_country,destination_country,merchant_category,device_type,device_trusted,cross_border,high_risk_country,velocity_score,fraud_label
0,2025-11-06 11:20:57,8493.46,INR,CARD,11,3,False,3407,4807,40,...,True,IN,IN,HEALTHCARE,IOS,True,False,False,0.821,0
1,2025-11-06 11:33:44,3099.33,INR,UPI,11,3,False,3128,570,6,...,False,IN,FR,FOOD,WEB,True,True,False,0.107,0
2,2025-11-06 11:35:21,10262.47,INR,BANK_TRANSFER,11,3,False,4327,1884,36,...,False,IN,IN,UTILITIES,LINUX,True,False,False,0.619,0
3,2025-11-06 11:36:22,3927.67,INR,CARD,11,3,False,2981,4375,9,...,True,IN,IN,ENTERTAINMENT,WEB,True,False,False,0.111,0
4,2025-11-06 11:37:17,3531.86,INR,CARD,11,3,False,2466,491,3,...,False,IN,IN,UTILITIES,WEB,True,False,False,0.027,0


In [6]:
# preparing raw features

binary_features = [

    "is_weekend",

    "is_new_receiver",

    "device_trusted",

    "cross_border",

    "high_risk_country"

]

TARGET = "fraud_label"

X = test_df.drop(columns=TARGET)

y = test_df[TARGET]

X = X.drop(
    columns="timestamp"
)

X[binary_features] = X[binary_features].astype("int8")

In [7]:
# production pipeline

X_processed = preprocessor.transform(X)

probabilities = model.predict_proba(
    X_processed
)[:,1]

In [8]:
# use production threshold

threshold = metadata["optimal_threshold"]

predictions = (

    probabilities >= threshold

).astype(int)

In [9]:
# sample predictions

results = pd.DataFrame({

    "Actual": y,

    "Fraud Probability": probabilities,

    "Prediction": predictions

})

results.head(20)

,Actual,Fraud Probability,Prediction
0,0,0.143624,0
1,0,0.003959,0
2,0,0.010653,0
3,0,0.010694,0
4,0,0.024443,0
5,0,0.026765,0
6,0,0.015821,0
7,0,0.009076,0
8,0,0.005978,0
9,0,0.005035,0


In [10]:
# demo of a single transaction

sample = X.iloc[[0]]

sample_processed = preprocessor.transform(sample)

probability = model.predict_proba(
    sample_processed
)[0,1]

prediction = int(

    probability >= threshold

)

print("="*60)

print("Single Transaction Prediction")

print("="*60)

print(f"Fraud Probability : {probability:.4f}")

print(f"Prediction        : {prediction}")

Single Transaction Prediction
Fraud Probability : 0.1436
Prediction        : 0


In [12]:
# deployment summary

print("="*70)

print("Production Export Completed Successfully")

print("="*70)

print()

print("Artifacts")

print(f"✓ {PIPELINE_PATH.name}")

print(f"✓ {MODEL_PATH.name}")

print(f"✓ {METADATA_PATH.name}")

print()

print(f"Model      : {metadata['model_name']}")

print(f"Version    : {metadata['model_version']}")

print(f"Threshold  : {metadata['optimal_threshold']:.2f}")

print(f"Features   : {metadata['feature_count']}")

print()

print("Ready for FastAPI Deployment")

print("="*70)

Production Export Completed Successfully

Artifacts
✓ preprocessing_pipeline.joblib
✓ best_model.joblib
✓ model_metadata.json

Model      : LGBMClassifier
Version    : 1.0.0
Threshold  : 0.40
Features   : 41

Ready for FastAPI Deployment
